# S03 — Linear regression in a small example
**Purpose:** a 20–30 minute lecturer demonstration or catch-up notebook. The main 90-minute student activity is
[Regression Quest](https://github.com/fajouri/msc-machine-learning/tree/master/sessions/S03/tutorials/01-ai-subscription-regression).
These are original synthetic teaching data, not real observations. This demonstration has no XP and is not an assessment solution.

Predict a fictional cost in EUR from a current usage measurement. Keep the final test separate.


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print('Python', sys.version.split()[0], 'NumPy', np.__version__, 'pandas', pd.__version__, 'sklearn', sklearn.__version__)
rng = np.random.default_rng(42)
usage = rng.uniform(1, 25, 120)
cost = 8 + 2.4*usage + rng.normal(0, 5, 120)
X = pd.DataFrame({'usage':usage})
y = pd.Series(cost, name='cost_EUR')
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.2, random_state=42)
print('Development / test:', len(X_dev), len(X_test))


## 1. The objective
For a teaching line `prediction = 10 + 2*usage`, calculate `residual = actual − prediction`.
SSE sums squared residuals. MSE divides SSE by the row count and has the same minimum for fixed n.
Run the example, then explain why RMSE and MAE are in euros while MSE is in squared euros.


In [ ]:
actual = np.array([20.,30.,40.])
predicted = np.array([18.,33.,39.])
residual = actual-predicted
print('Residuals:', residual)
print('SSE:', np.sum(residual**2), 'MSE:', np.mean(residual**2))
print('MAE:', np.mean(np.abs(residual)), 'RMSE:', np.sqrt(np.mean(residual**2)))


## 2. Development cross-validation
Each pipeline fits its own imputer and scaler inside the fold. Compare identical five-fold splits.
The MAE scorer returns negative values, which we negate for readable euro errors.
Five folds fit five fresh estimators per candidate. The seed controls splits, not epochs.


In [ ]:
def pipeline(estimator):
    return Pipeline([('impute',SimpleImputer(strategy='median')), ('scale',StandardScaler()), ('model',estimator)])
models = {'Mean baseline':pipeline(DummyRegressor()), 'Linear regression':pipeline(LinearRegression())}
folds = list(KFold(n_splits=5,shuffle=True,random_state=42).split(X_dev))
results = {}
for name,model in models.items():
    scores = cross_validate(model,X_dev,y_dev,cv=folds,scoring='neg_mean_absolute_error',error_score='raise')
    errors = -scores['test_score']
    results[name] = {'CV_MAE_EUR':errors.mean(),'fold_sd_EUR':errors.std(ddof=1)}
comparison = pd.DataFrame(results).T
display(comparison)
# Predeclared rule: minimum CV MAE, with mean baseline first if exactly tied.
chosen = min(models, key=lambda name:results[name]['CV_MAE_EUR'])
print('Chosen using development CV:',chosen)


## 3. One final test check
Refit the chosen model on development data, then evaluate the reserved test once.
Interpret the observed result without changing the model using this test feedback.
R² is dimensionless and may be negative. MAE does not bound an individual future error.


In [ ]:
final_model = models[chosen].fit(X_dev,y_dev)
pred = final_model.predict(X_test)
print({'test_MAE_EUR':mean_absolute_error(y_test,pred),
       'test_RMSE_EUR':np.sqrt(mean_squared_error(y_test,pred)), 'test_R2':r2_score(y_test,pred)})


## 4. A line with interpretable units
For interpretation, fit a separate unscaled single-input line on development data only.
Its slope is euros per usage unit. This explanatory fit does not revise the locked candidate or test result.
The plot displays development observations and the line. The association is synthetic and not causal.


In [ ]:
explanation_model = LinearRegression().fit(X_dev,y_dev)
grid = pd.DataFrame({'usage':np.linspace(X_dev.usage.min(),X_dev.usage.max(),100)})
plt.figure(figsize=(7,4))
plt.scatter(X_dev.usage,y_dev,label='Synthetic development rows',alpha=.6)
plt.plot(grid.usage,explanation_model.predict(grid),color='black',label='Development least-squares line')
plt.xlabel('Current usage (units)')
plt.ylabel('Fictional cost (EUR)')
plt.title('A single-input regression line')
plt.legend()
plt.tight_layout()
plt.show()
print('Slope, EUR per usage unit:',explanation_model.coef_[0])
print('Intercept, EUR:',explanation_model.intercept_)
print('Text alternative: the synthetic development costs tend to rise with usage, with visible noise around the fitted line.')


## Discussion
Why might a random split fail for repeated users or later calendar periods? What would a curved residual pattern suggest?
How do the coefficients' units change if you standardise the input? Complete the separate Regression Quest for step-by-step tasks and XP.

References: [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html),
[cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html),
[common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html).
